**NovaPay – Feature Engineering Notebook**

Fraud Detection Feature Construction & Model-Ready Dataset Preparation

**1. Introduction**

The goal of this notebook is to transform the cleaned NovaPay dataset into a feature-enriched, model-ready format.
Fraud detection relies heavily on engineered features because raw transactional fields rarely give enough insight to distinguish malicious vs legitimate behavior.

This notebook will create:

* Time-based features

* Customer behavior metrics

* Transactional derived variables

* Velocity and frequency signals

* Risk-tier transformations

* Device/IP-based intelligence

* Encoded categorical features

* Final ML-ready dataset

**2. Load Data**

In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NOVA PAY/nova_pay_with_time_features.csv", parse_dates=["timestamp"])

df.head()


,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,fee,...,kyc_tier,account_age_days,device_trust_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,is_fraud,month
0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,us,usd,cad,atm,278.19,278.19,4.25,...,standard,263,0.522,0,0.223,0,0,0.0,0,2022-10
1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,ca,cad,mxn,web,208.51,154.29,4.24,...,standard,947,0.475,0,0.268,0,1,0.0,0,2022-10
2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,us,usd,cny,mobile,160.33,160.33,2.70,...,enhanced,367,0.939,0,0.176,0,0,0.0,0,2022-10
3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,us,usd,eur,mobile,59.41,59.41,2.22,...,standard,147,0.551,0,0.391,0,0,0.0,0,2022-10
4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,us,usd,inr,mobile,200.96,200.96,3.61,...,enhanced,257,0.894,0,0.257,0,0,0.0,0,2022-10


**3. Time-Based Feature Engineering**

**3.1 Extract timestamps**

In [23]:
df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day
df["weekday"] = df["timestamp"].dt.weekday
df["week"] = df["timestamp"].dt.isocalendar().week
df["month"] = df["timestamp"].dt.month
df["year"] = df["timestamp"].dt.year


**3.2 Weekend vs Weekday Flag**

Fraud tends to increase on weekends.

In [24]:
df["is_weekend"] = df["weekday"].isin([5,6]).astype(int)


**3.3 Time-of-Day Bands**

In [25]:
def time_band(h):
    if h < 6: return "late_night"
    if h < 12: return "morning"
    if h < 18: return "afternoon"
    return "evening"

df["time_band"] = df["hour"].apply(time_band)


**4. Transaction-Based Feature Engineering**

**4.1 Amount-Based Features**

In [26]:
df["amount_log"] = np.log1p(df["amount_usd"])  # reduces skew
df["fee_ratio"] = df["fee"] / df["amount_usd"]


**4.2 Currency Path Feature**

In [27]:
df["currency_pair"] = df["source_currency"] + "_" + df["dest_currency"]


**4.3 High-Risk Corridor Flag**

In [28]:
df["high_corridor"] = (df["corridor_risk"] > df["corridor_risk"].median()).astype(int)


**5. Customer Behavior Features**

**5.1 Total transactions per customer**

In [29]:
df["customer_txn_count"] = df.groupby("customer_id")["transaction_id"].transform("count")


**5.2 Average amount per customer**

In [30]:
df["customer_avg_amount"] = df.groupby("customer_id")["amount_usd"].transform("mean")


**5.3 Customer total USD volume**

In [31]:
df["customer_total_usd"] = df.groupby("customer_id")["amount_usd"].transform("sum")


**5.4 Customer risk profile**

In [32]:
df["customer_avg_ip_risk"] = df.groupby("customer_id")["ip_risk_score"].transform("mean")
df["customer_avg_trust_score"] = df.groupby("customer_id")["device_trust_score"].transform("mean")


**5.5 Returning vs New Customer**

In [33]:
df["is_returning_customer"] = (df["customer_txn_count"] > 1).astype(int)


**6. Device & IP Intelligence Features**

**6.1 New Device Flag**

In [34]:
df["new_device"] = df["new_device"].astype(int)


**6.2 Mismatch Location Risk**

In [35]:
df["location_mismatch"] = df["location_mismatch"].astype(int)


**6.3 IP Address Frequency**

Fraudsters often reuse the same IP for multiple victims.

In [36]:
df["ip_usage_count"] = df.groupby("ip_address")["transaction_id"].transform("count")


6.4 Rare IP Flag

In [37]:
df["rare_ip"] = (df["ip_usage_count"] == 1).astype(int)


**7. Velocity Features**

**7.1 Customer transaction velocity**

In [38]:
df = df.sort_values(["customer_id", "timestamp"]).reset_index(drop=True)

df["prev_txn_time"] = df.groupby("customer_id")["timestamp"].shift(1)
df["time_since_last_txn"] = (df["timestamp"] - df["prev_txn_time"]).dt.total_seconds()

df["time_since_last_txn"].fillna(df["time_since_last_txn"].median(), inplace=True)


/tmp/ipython-input-4184299059.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["time_since_last_txn"].fillna(df["time_since_last_txn"].median(), inplace=True)


Low “time since last transaction” = high fraud likelihood.

**7.2 Velocity ratio**

In [39]:
df["velocity_ratio"] = df["txn_velocity_1h"] / (df["txn_velocity_24h"] + 1)


**8. Encoding Categorical Features**

8.1 Label Encoding for tree-based models

In [40]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    "home_country", "source_currency", "dest_currency", "channel",
    "kyc_tier", "ip_country", "currency_pair", "time_band"
]

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col].astype(str))


**9. Outlier Treatment**

**Clip extreme values**

In [41]:
df["amount_usd"] = df["amount_usd"].clip(lower=0, upper=df["amount_usd"].quantile(0.99))
df["device_trust_score"] = df["device_trust_score"].clip(0, 1)
df["ip_risk_score"] = df["ip_risk_score"].clip(0, 1)


**10. Final Feature Set & Export**

**10.1 Drop non-useful identifiers for modeling**

In [42]:
df_model = df.drop(columns=["transaction_id", "customer_id", "device_id", "timestamp", "ip_address"])


**10.2 Save dataset**

In [43]:
df_model.to_csv("novapay_features_engineered.csv", index=False)
print("Feature engineered dataset saved.")


Feature engineered dataset saved.


**11. Summary of Key Engineered Features**

This notebook created:

**Time Features**

hour, weekday, month, is_weekend, time_band

**Behavioral Features**

customer_txn_count, avg amount, avg risk, returning customer

**Risk Features**

high_corridor, ip_usage_count, rare_ip

**Velocity Features**

time_since_last_txn, velocity_ratio

**Transactional Features**

amount_log, fee_ratio, currency_pair

**Categorical Encodings**

label encoding for all major categorical fields

**Normalizations & Corrections**

value clipping, trust score correction, outlier handling

